# 14 -- Calibración conformal por estrato del cuantil relativo (intermitente)

`13` mostró que en `intermitente` un cuantil sobre el target relativo (`y/base`) reduce el costo frente al de producción (+6.4% a alpha 0.893), pero **subcubre en la cabeza**
(cobertura 0.82 con objetivo 0.893). Aquí se prueba una recalibración conformal (split-CQR) **por estrato**:

- Para cada fold: entrenamiento con purge; separación cronológica fit/calib 80/20 **purgada** (la ventana objetivo de fit no toca calib).
- Modelo LightGBM cuantílico (alpha 0.893) sobre `y/base` entrenado en `fit`; el residual `y/base - q` en `calib` se resume, **por estrato as-of**, con el cuantil conformal
  `ceil((n+1)·alpha)/n`; el modelo final se entrena con todo el entrenamiento y la predicción es `base · (q_relativo + offset_estrato)`.
- Variantes: `sin_cal` (modelo sin corregir), `cal_global` (un solo offset para toda la rama) y `cal_estrato` (un offset por estrato).
- Métrica: costo asimétrico real (Cu=0.25, Co=0.03) frente a la base por estrato, y cobertura por estrato. Hiperparámetros de `MODELOS_POR_RAMA['intermitente']`, sin búsqueda.

Regla mecánica: `cal_estrato` se adopta si (a) mejora el costo de `sin_cal` con IC95 > 0 en los folds 2-4, (b) el fold 5 no lo contradice y (c) la cobertura de **cada** estrato queda a ≤ 0.03 del objetivo.

Cadena: ... -> `13_cuantiles_y_costos` -> **`14_calibracion_conformal_estrato`**

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella, purgar_entrenamiento, fin_ventana_de
import evaluar_nivel1 as ev
from exportar_modelos_nivel1 import FEATURES as F1, MODELOS_POR_RAMA, RAMA_A_CONDICION, COSTOS_SUPUESTOS

import json, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
warnings.filterwarnings('ignore')

pd.set_option('display.width', 230)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

EXPERIMENTO = 'EDA-fix-nivel1-14-calibracion-conformal-estrato'
Y, PISO, H = 'target_demanda_15d', 'pred_media_movil_15d', 15
LLAVES = ['codigo_item', 'sucursal', 'fecha_origen']
ALPHA = 0.893
CU, CO = COSTOS_SUPUESTOS['no_perecedero']['Cu'], COSTOS_SUPUESTOS['no_perecedero']['Co']
ESTRATOS = ['cabeza', 'medio', 'cola']
FRACCION_CALIB, MIN_CALIB_ESTRATO, TOL_COBERTURA = 0.2, 30, 0.03
HP = MODELOS_POR_RAMA['intermitente']['hparams']
B, SEMILLA = 2000, 0

verificar_huella('huella_07.json', ['matriz_as_of.parquet'])
verificar_huella('huella_12.json', ['estratos_por_fold_12.parquet'])
verificar_huella('huella_13.json', ['preds_cuantiles_13.parquet'])
verificar_huella('huella_08.json', ['evaluacion_nivel_1.parquet'])
m = pd.read_parquet(f'{DW}/matriz_as_of.parquet')
m['fin_ventana'] = fin_ventana_de(m, H)
estr = pd.read_parquet(f'{DW}/estratos_por_fold_12.parquet')
estr['codigo_item'], estr['sucursal'] = estr['codigo_item'].astype(str), estr['sucursal'].astype(str)

def etiqueta_estrato(df, f):
    e = estr[estr['fold_id'] == f].set_index(['codigo_item', 'sucursal'])['estrato']
    return e.reindex(pd.MultiIndex.from_arrays([df['codigo_item'].astype(str), df['sucursal'].astype(str)])).to_numpy()

dr = m.loc[RAMA_A_CONDICION['intermitente'](m)]
print(dr.shape, '| alpha', ALPHA, '| costos', CU, CO)

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 20:03:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07.json.
Huella verificada OK contra huella_12.json.
Huella verificada OK contra huella_13.json.
Huella verificada OK contra huella_08.json.


(257894, 28) | alpha 0.893 | costos 0.25 0.03


## 1. Walk-forward con calibración conformal por estrato

In [2]:
def lgb_q(X, y, alpha, hp):
    mod = lgb.LGBMRegressor(objective='quantile', alpha=alpha, subsample_freq=1, random_state=42, verbose=-1, **hp)
    mod.fit(X, y)
    return mod

def offset_conformal(res_rel, alpha):
    n = len(res_rel)
    return float(np.quantile(res_rel, min(1.0, np.ceil((n + 1) * alpha) / n)))

partes, offsets, t0 = [], [], time.time()
for f in [2, 3, 4, 5]:
    tr = purgar_entrenamiento(dr, f, verbose=False)
    te = dr[dr['fold_id'] == f]
    base_tr, base_te = tr['nivel_medio_60d'] * H + 1.0, te['nivel_medio_60d'] * H + 1.0
    e_tr, e_te = etiqueta_estrato(tr, f), etiqueta_estrato(te, f)
    # separación cronológica fit/calib, con purge de la ventana objetivo
    fechas = np.sort(tr['fecha_origen'].unique())
    corte = fechas[int(len(fechas) * (1 - FRACCION_CALIB))]
    en_fit = ((tr['fecha_origen'] < corte) & (tr['fin_ventana'] < corte)).to_numpy()
    en_cal = (tr['fecha_origen'] >= corte).to_numpy()
    mod_fit = lgb_q(tr.loc[en_fit, F1], tr.loc[en_fit, Y] / base_tr[en_fit], ALPHA, HP)
    res_rel = (tr.loc[en_cal, Y] / base_tr[en_cal]).to_numpy() - mod_fit.predict(tr.loc[en_cal, F1])
    off_global = offset_conformal(res_rel, ALPHA)
    off_est = {}
    for e in ESTRATOS:
        r = res_rel[e_tr[en_cal] == e]
        off_est[e] = offset_conformal(r, ALPHA) if len(r) >= MIN_CALIB_ESTRATO else off_global
        offsets.append(dict(fold_id=f, estrato=e, n_calib=int(len(r)), offset_rel=off_est[e], offset_global=off_global))
    mod_full = lgb_q(tr[F1], tr[Y] / base_tr, ALPHA, HP)
    q_rel = mod_full.predict(te[F1])
    blk = te[LLAVES + ['fold_id']].copy()
    blk['y'], blk['estrato'] = te[Y].to_numpy(), e_te
    blk['q_sin_cal'] = np.clip(q_rel * base_te.to_numpy(), 0, None)
    blk['q_cal_global'] = np.clip((q_rel + off_global) * base_te.to_numpy(), 0, None)
    blk['q_cal_estrato'] = np.clip((q_rel + pd.Series(e_te).map(off_est).to_numpy()) * base_te.to_numpy(), 0, None)
    partes.append(blk)
    print(f'fold {f}: train={len(tr):,} (fit={int(en_fit.sum()):,}, calib={int(en_cal.sum()):,}) test={len(te):,} '
          f'| offsets (rel): ' + ', '.join(f'{e} {off_est[e]:+.2f}' for e in ESTRATOS) + f' | global {off_global:+.2f} ({time.time()-t0:,.0f}s)')
res = pd.concat(partes, ignore_index=True)
for c in ('codigo_item', 'sucursal'):
    res[c] = res[c].astype(str)
assert res['estrato'].notna().all()
print(pd.DataFrame(offsets).round(3).to_string(index=False))

fold 2: train=20,152 (fit=10,076, calib=5,038) test=68,198 | offsets (rel): cabeza +0.14, medio +0.09, cola +0.15 | global +0.13 (5s)


fold 3: train=88,142 (fit=61,912, calib=20,984) test=28,850 | offsets (rel): cabeza +0.11, medio +0.12, cola +0.06 | global +0.08 (13s)


fold 4: train=116,468 (fit=82,896, calib=28,326) test=59,600 | offsets (rel): cabeza +0.25, medio +0.17, cola +0.06 | global +0.09 (21s)


fold 5: train=175,878 (fit=128,198, calib=41,720) test=76,056 | offsets (rel): cabeza +0.21, medio +0.11, cola +0.01 | global +0.04 (30s)
 fold_id estrato  n_calib  offset_rel  offset_global
       2  cabeza      138       0.141          0.132
       2   medio     1125       0.095          0.132
       2    cola     3775       0.149          0.132
       3  cabeza      540       0.108          0.081
       3   medio     4532       0.120          0.081
       3    cola    15912       0.063          0.081
       4  cabeza      673       0.247          0.094
       4   medio     5740       0.173          0.094
       4    cola    21913       0.058          0.094
       5  cabeza      980       0.211          0.037
       5   medio     8218       0.114          0.037
       5    cola    32522       0.006          0.037


## 2. Control de replicación y comparación

`sin_cal` debe reproducir el `qm_0.893` de `13` (mismo modelo y entrenamiento). Luego se agregan la base por estrato (`qe`) y el cuantil de producción de 08.

In [3]:
p13 = pd.read_parquet(f'{DW}/preds_cuantiles_13.parquet')
p13 = p13[p13['rama'] == 'intermitente'][LLAVES + ['piso', 'qm_0.893', 'qe_0.893', 'qb_0.893']]
prod = pd.read_parquet(f'{DW}/evaluacion_nivel_1.parquet').dropna(subset=['rama'])
prod = prod[prod['rama'] == 'intermitente'][LLAVES + ['pred_q_negocio']]
for d in (p13, prod):
    for c in ('codigo_item', 'sucursal'):
        d[c] = d[c].astype(str)
res = res.merge(p13, on=LLAVES, how='left', validate='one_to_one').merge(prod, on=LLAVES, how='left', validate='one_to_one')
assert res[['qm_0.893', 'qe_0.893', 'pred_q_negocio']].notna().all().all()
dif = float((res['q_sin_cal'] - res['qm_0.893']).abs().max())
print(f'diferencia máxima sin_cal vs 13: {dif:.6f}')
assert dif < 1e-3, 'no reproduce 13'
res = res.rename(columns={'qe_0.893': 'q_base_estrato', 'pred_q_negocio': 'q_produccion'})
VARIANTES = ['q_produccion', 'q_sin_cal', 'q_cal_global', 'q_cal_estrato']

diferencia máxima sin_cal vs 13: 0.000000


## 3. Cobertura y costo por estrato (alpha 0.893, Cu=0.25, Co=0.03)

In [4]:
def costo(g, col):
    return ev.costo_asimetrico(g['y'], g[col], CU, CO)

filas = []
for est in ESTRATOS + ['todos']:
    g = res if est == 'todos' else res[res['estrato'] == est]
    cb = costo(g, 'q_base_estrato')
    for v in ['q_base_estrato'] + VARIANTES:
        r = ev.mejora_perdida_con_ic(g, costo(g, v), cb, B, SEMILLA)
        fila = dict(estrato=est, variante=v[2:], n_filas=len(g), cobertura=float((g['y'] <= g[v]).mean()),
                    desvio=float((g['y'] <= g[v]).mean() - ALPHA), costo_por_unidad=r['perdida_modelo'] / g['y'].sum(),
                    mejora_vs_base=r['mejora_rel'], lo=r['ic_bajo'], hi=r['ic_alto'])
        if v not in ('q_base_estrato', 'q_sin_cal'):
            for etiqueta, sub in (('f2a4', g[g['fold_id'].isin(ev.FOLDS_BUSQUEDA)]), ('f5', g[g['fold_id'] == ev.FOLD_HOLDOUT])):
                q = ev.mejora_perdida_con_ic(sub, costo(sub, v), costo(sub, 'q_sin_cal'), B, SEMILLA)
                fila.update({f'vs_sincal_{etiqueta}': q['mejora_rel'], f'vs_sincal_{etiqueta}_lo': q['ic_bajo'], f'vs_sincal_{etiqueta}_hi': q['ic_alto']})
        filas.append(fila)
tab = pd.DataFrame(filas)
for est in ESTRATOS + ['todos']:
    print(f'\n=== {est.upper()} ===')
    print(tab[tab['estrato'] == est].drop(columns=['estrato', 'n_filas']).round(3).to_string(index=False))


=== CABEZA ===
    variante  cobertura  desvio  costo_por_unidad  mejora_vs_base     lo     hi  vs_sincal_f2a4  vs_sincal_f2a4_lo  vs_sincal_f2a4_hi  vs_sincal_f5  vs_sincal_f5_lo  vs_sincal_f5_hi
base_estrato      0.896   0.003             0.029           0.000  0.000  0.000             NaN                NaN                NaN           NaN              NaN              NaN
  produccion      0.816  -0.077             0.033          -0.152 -0.328 -0.014          -0.337             -0.760             -0.073         0.004           -0.029            0.039
     sin_cal      0.820  -0.073             0.027           0.078  0.018  0.158             NaN                NaN                NaN           NaN              NaN              NaN
  cal_global      0.869  -0.024             0.026           0.098  0.059  0.150           0.032             -0.006              0.060        -0.007           -0.020            0.003
 cal_estrato      0.895   0.002             0.027           0.056  0.024  

## 4. Decisión mecánica, artefactos y registro en MLflow

In [5]:
d = tab[tab['variante'] == 'cal_estrato'].set_index('estrato')
cob_ok = {e: bool(abs(d.loc[e, 'desvio']) <= TOL_COBERTURA) for e in ESTRATOS}
a = d.loc['todos']
adopta = bool(a['vs_sincal_f2a4_lo'] > 0 and a['vs_sincal_f5'] > 0 and all(cob_ok.values()))
print('cobertura por estrato (cal_estrato):', {e: round(d.loc[e, 'cobertura'], 3) for e in ESTRATOS}, '| dentro de ±0.03:', cob_ok)
print(f"vs sin_cal: folds 2-4 {a['vs_sincal_f2a4']:+.2%} IC [{a['vs_sincal_f2a4_lo']:+.2%}, {a['vs_sincal_f2a4_hi']:+.2%}] | fold 5 {a['vs_sincal_f5']:+.2%}")
print('DECISIÓN cal_estrato:', 'ADOPTA' if adopta else 'NO adopta')
dec = {'notebook': '14_calibracion_conformal_estrato', 'fecha_generacion': pd.Timestamp.now().isoformat(), 'alpha': ALPHA, 'costos': {'Cu': CU, 'Co': CO},
       'hiperparametros': 'fijos (MODELOS_POR_RAMA intermitente)', 'fraccion_calib': FRACCION_CALIB, 'min_calib_estrato': MIN_CALIB_ESTRATO, 'tolerancia_cobertura': TOL_COBERTURA,
       'politica_holdout': ev.POLITICA_HOLDOUT, 'offsets': pd.DataFrame(offsets).round(4).to_dict('records'),
       'cobertura_por_estrato': {v: {e: float(tab[(tab.variante == v) & (tab.estrato == e)]['cobertura'].iloc[0]) for e in ESTRATOS + ['todos']}
                                 for v in ['produccion', 'sin_cal', 'cal_global', 'cal_estrato', 'base_estrato']},
       'cal_estrato': {'cobertura_dentro_tolerancia': cob_ok, 'vs_sin_cal_f2a4': float(a['vs_sincal_f2a4']),
                       'ic_f2a4': [float(a['vs_sincal_f2a4_lo']), float(a['vs_sincal_f2a4_hi'])], 'vs_sin_cal_f5': float(a['vs_sincal_f5']),
                       'mejora_vs_base': float(a['mejora_vs_base']), 'ic_vs_base': [float(a['lo']), float(a['hi'])], 'adopta': adopta}}
with open(f'{DW}/decision_14.json', 'w') as f:
    json.dump(dec, f, indent=2, ensure_ascii=False)
tab.to_csv(f'{DW}/calibracion_conformal_14.csv', index=False)
res.to_parquet(f'{DW}/preds_calibracion_14.parquet', index=False)
huella = {'notebook': '14_calibracion_conformal_estrato', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of.parquet', 'estratos_por_fold_12.parquet', 'preds_cuantiles_13.parquet', 'evaluacion_nivel_1.parquet']),
          'salidas': registrar_huella(['decision_14.json', 'calibracion_conformal_14.csv', 'preds_calibracion_14.parquet'])}
with open(f'{DW}/huella_14.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
for _, r in tab.iterrows():
    ev.registrar_mlflow(EXPERIMENTO, f"{r['estrato']}__{r['variante']}",
        tags={'fase': 'E', 'notebook': '14', 'estrato': r['estrato'], 'variante': r['variante'], 'rama_git': 'fix/eda-modelado-nivel1', 'adopta': str(adopta)},
        params={'alpha': ALPHA, 'Cu': CU, 'Co': CO, 'fraccion_calib': FRACCION_CALIB, 'hiperparametros': 'fijos'},
        metrics={k: r[k] for k in ['n_filas', 'cobertura', 'desvio', 'costo_por_unidad', 'mejora_vs_base', 'lo', 'hi', 'vs_sincal_f2a4', 'vs_sincal_f2a4_lo', 'vs_sincal_f5'] if k in r and pd.notna(r[k])})
print(f'{len(tab)} runs registrados en {EXPERIMENTO}')

cobertura por estrato (cal_estrato): {'cabeza': np.float64(0.895), 'medio': np.float64(0.901), 'cola': np.float64(0.906)} | dentro de ±0.03: {'cabeza': True, 'medio': True, 'cola': True}
vs sin_cal: folds 2-4 +0.06% IC [-1.14%, +0.96%] | fold 5 -3.45%
DECISIÓN cal_estrato: NO adopta


2026/09/24 20:04:13 INFO mlflow.tracking.fluent: Experiment with name 'EDA-fix-nivel1-14-calibracion-conformal-estrato' does not exist. Creating a new experiment.


🏃 View run cabeza__base_estrato at: http://127.0.0.1:5000/#/experiments/21/runs/b6f37f9026a8459294540596ccb02ac2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


🏃 View run cabeza__produccion at: http://127.0.0.1:5000/#/experiments/21/runs/cc38e03fa1fd400aae8ce6b1b22802ae
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


🏃 View run cabeza__sin_cal at: http://127.0.0.1:5000/#/experiments/21/runs/b643750c11a741ebae92bcba2d95de9c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


...

🏃 View run todos__base_estrato at: http://127.0.0.1:5000/#/experiments/21/runs/35a28ba6b2574bd3bcaf30d37cdf0a35
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


🏃 View run todos__produccion at: http://127.0.0.1:5000/#/experiments/21/runs/b8def7e6b4734492b06ba214cc75f50f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


🏃 View run todos__sin_cal at: http://127.0.0.1:5000/#/experiments/21/runs/f1bccb1b1f644cc8a7eb368017765fdc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


🏃 View run todos__cal_global at: http://127.0.0.1:5000/#/experiments/21/runs/dfa77b0190c54733b41bb323c05aaea9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21


🏃 View run todos__cal_estrato at: http://127.0.0.1:5000/#/experiments/21/runs/69aee88d9f384f27907b2298480aa906
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/21
20 runs registrados en EDA-fix-nivel1-14-calibracion-conformal-estrato
